In [17]:
import random
from reedsolo import RSCodec

class ReedSolomonSimulator:
    def __init__(self):
        self.rs_codes = {
            'RS(7,4)': {'codec': RSCodec(3), 'n': 7, 'k': 4, 't': 1},
            'RS(15,11)': {'codec': RSCodec(4), 'n': 15, 'k': 11, 't': 2},
            'RS(31,25)': {'codec': RSCodec(6), 'n': 31, 'k': 25, 't': 3},
        }
        self.results = []

    def display_hex_with_colors(self, data, error_positions=None, corrected_positions=None):
        result = []
        for i, byte in enumerate(data):
            if error_positions and i in error_positions:
                result.append(f"\033[91m{byte:02x}\033[0m")  # Red for errors
            elif corrected_positions and i in corrected_positions:
                result.append(f"\033[92m{byte:02x}\033[0m")  # Green for corrected
            else:
                result.append(f"{byte:02x}")
        return ' '.join(result)

    def run_test(self, test_num, rs_name, error_count):
        config = self.rs_codes[rs_name]
        rsc = config['codec']
        n, k, t = config['n'], config['k'], config['t']
        
        original = bytes([random.randint(0, 255) for _ in range(k)])
        
        print(f"\n{'━'*70}")
        print(f"📊 TEST {test_num}: {rs_name} (t={t}, can correct {t} errors)")
        print(f"{'━'*70}")
        
        print(f"\n📤 SENDER:")
        print(f"Original data ({k} bytes): {original.hex()}")
        
        encoded_full = rsc.encode(original)
        encoded = encoded_full[:n]
        print(f"Encoded ({n} symbols): {self.display_hex_with_colors(encoded)}")
        print(f"  Data part: {self.display_hex_with_colors(encoded[:k])}")
        print(f"  Parity part: {self.display_hex_with_colors(encoded[k:])}")
        
        corrupted = bytearray(encoded)
        error_positions = random.sample(range(n), min(error_count, n))
        
        error_details = []
        for pos in error_positions:
            old = corrupted[pos]
            new = (old + random.randint(1, 100)) % 256
            corrupted[pos] = new
            error_details.append(f"{pos}: {old:02x}→{new:02x}")
        
        print(f"\n💥 CHANNEL:")
        print(f"Added {len(error_positions)} errors: {', '.join(error_details)}")
        print(f"Received data: {self.display_hex_with_colors(corrupted, error_positions)}")
        
        print(f"\n🔍 RECEIVER - Decoding Steps:")
        print("1️⃣ Syndrome Calculation:")
        print("   Sᵢ = Σ rⱼ × (αʲ)ⁱ   (in GF(2⁸))")
        print("   If all Sᵢ = 0 → no errors")
        print("   If Sᵢ ≠ 0 → errors detected")
        
        try:
            print("\n2️⃣ Error Location Finding:")
            print("   Solving: Λ(x) = 1 + Λ₁x + Λ₂x² + ...")
            print("   Roots of Λ(x) give error positions")
            
            padded = bytes(corrupted) + b'\x00' * (255 - n)
            decoded_result = rsc.decode(padded)
            corrected_data = decoded_result[0]
            corrected_symbols = corrected_data[:n]
            decoded = corrected_data[:k]
            
            corrected_positions = []
            for i in range(n):
                if corrected_symbols[i] != corrupted[i]:
                    corrected_positions.append(i)
            
            print(f"\n3️⃣ Error Correction:")
            print(f"   Found errors at positions: {sorted(corrected_positions)}")
            
            is_correct = decoded == original
            
            print(f"\n✅ DECODING COMPLETE:")
            print(f"Corrected data: {self.display_hex_with_colors(corrected_symbols, error_positions, corrected_positions)}")
            print(f"Decoded data: {decoded.hex()}")
            print(f"Match original: {'✅ YES' if is_correct else '❌ NO'}")
            
            success = True
            
        except Exception as e:
            print(f"\n❌ DECODING FAILED:")
            print(f"   Too many errors! Can only correct {t} errors, but {error_count} occurred")
            decoded = bytes(corrupted)[:k]
            is_correct = False
            success = False
            corrected_positions = []
        
        print(f"\n🎯 RESULT: {'SUCCESS' if is_correct else 'FAILURE'}")
        print(f"   Errors added: {error_count}")
        print(f"   Max correctable: {t}")
        print(f"   {'✓ Within capacity' if error_count <= t else '✗ Exceeds capacity'}")
        
        self.results.append({
            'test': test_num,
            'code': rs_name,
            'errors': error_count,
            'max_t': t,
            'success': success,
            'correct': is_correct,
            'error_positions': sorted(error_positions),
            'corrected_positions': sorted(corrected_positions)
        })
        
        return is_correct

print("╔══════════════════════════════════════════════════════════╗")
print("║           REED-SOLOMON ERROR CORRECTION DEMO            ║")
print("╚══════════════════════════════════════════════════════════╝")

simulator = ReedSolomonSimulator()

print("\n🔧 Testing Reed-Solomon Codes")
print("RS(n,k): n=total symbols, k=data symbols")
print("t = (n-k)/2 = max correctable errors")
print("Color legend: \033[91mRed\033[0m = Error, \033[92mGreen\033[0m = Corrected")

tests = [
    (1, 'RS(7,4)', 0),
    (2, 'RS(7,4)', 1),
    (3, 'RS(7,4)', 1),
    (4, 'RS(7,4)', 2),
    (5, 'RS(15,11)', 0),
    (6, 'RS(15,11)', 1),
    (7, 'RS(15,11)', 2),
    (8, 'RS(15,11)', 3),
    (9, 'RS(31,25)', 0),
    (10, 'RS(31,25)', 2),
    (11, 'RS(31,25)', 3),
    (12, 'RS(31,25)', 4),
]

for test_num, rs_type, errors in tests:
    simulator.run_test(test_num, rs_type, errors)

print(f"\n{'━'*70}")
print("📈 PERFORMANCE ANALYSIS")
print(f"{'━'*70}")

print("\n🎯 Success Rates by Code Type:")
print("="*70)

for code_name in simulator.rs_codes.keys():
    code_tests = [r for r in simulator.results if r['code'] == code_name]
    if code_tests:
        correct = sum(1 for r in code_tests if r['correct'])
        total = len(code_tests)
        percentage = (correct / total) * 100
        
        bars = int(percentage / 2)
        bar = '█' * bars + '░' * (50 - bars)
        
        t = code_tests[0]['max_t']
        print(f"{code_name:10} [{bar}] {correct}/{total} ({percentage:.0f}%)")
        print(f"           t={t}, Tested with errors: {[r['errors'] for r in code_tests]}")

print("\n" + "="*70)
print("📋 DETAILED RESULTS")
print("="*70)

print(f"{'Test':5} {'Code':10} {'Errors':8} {'t':3} {'Decoded':8} {'Result':12}")
print("-" * 70)

for result in simulator.results:
    test = result['test']
    code = result['code']
    errors = result['errors']
    t = result['max_t']
    decoded = 'SUCCESS' if result['success'] else 'FAILED'
    result_str = '✅ CORRECT' if result['correct'] else '❌ FAILED'
    
    print(f"{test:5} {code:10} {errors:8} {t:3} {decoded:8} {result_str:12}")

print("="*70)

print(f"\n{'━'*70}")
print("💡 KEY INSIGHTS")
print(f"{'━'*70}")

print("""
• RS(7,4): 4 data + 3 parity → t=1 (corrects 1 error)
• RS(15,11): 11 data + 4 parity → t=2 (corrects 2 errors)
• RS(31,25): 25 data + 6 parity → t=3 (corrects 3 errors)
• Decoder steps: 1) Syndrome calc 2) Find errors 3) Correct errors
• Works when errors ≤ t
• Fails when errors > t
• Real applications: CDs, DVDs, QR codes, satellite comm
""")

print(f"\n{'━'*70}")
print("🔬 ERROR ANALYSIS")
print(f"{'━'*70}")

print("Error positions vs corrected positions:")
for result in simulator.results:
    if result['error_positions']:
        print(f"Test {result['test']}:")
        print(f"  Actual errors:   {result['error_positions']}")
        print(f"  Found errors:    {result['corrected_positions']}")
        match = result['error_positions'] == result['corrected_positions']
        print(f"  Match: {'✅ Perfect' if match else '❌ Mismatch' if result['corrected_positions'] else '❌ Failed'}")



╔══════════════════════════════════════════════════════════╗
║           REED-SOLOMON ERROR CORRECTION DEMO            ║
╚══════════════════════════════════════════════════════════╝

🔧 Testing Reed-Solomon Codes
RS(n,k): n=total symbols, k=data symbols
t = (n-k)/2 = max correctable errors
Color legend: Red = Error, Green = Corrected

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 TEST 1: RS(7,4) (t=1, can correct 1 errors)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📤 SENDER:
Original data (4 bytes): c7c21a83
Encoded (7 symbols): c7 c2 1a 83 f4 9f f7
  Data part: c7 c2 1a 83
  Parity part: f4 9f f7

💥 CHANNEL:
Added 0 errors: 
Received data: c7 c2 1a 83 f4 9f f7

🔍 RECEIVER - Decoding Steps:
1️⃣ Syndrome Calculation:
   Sᵢ = Σ rⱼ × (αʲ)ⁱ   (in GF(2⁸))
   If all Sᵢ = 0 → no errors
   If Sᵢ ≠ 0 → errors detected

2️⃣ Error Location Finding:
   Solving: Λ(x) = 1 + Λ₁x + Λ₂x² + ...
   Roots of Λ(x) give error positions

3️⃣ Error Correc